# Quantitative End-to-End Evaluation

This notebook runs the quantitative evaluation of the complete grounded language pipeline on the commands defined in `evaluation_scenarios.json`. It compares the symbolic Stage 1 parser, the two learned Stage 2 parsers, and each parser combined with the Stage 3 Hugging Face model checker:
1. rule-based parser
2. rule-based parser with Stage 3
3. intent-classifier parser
4. intent-classifier parser with Stage 3
5. slot-tagger parser
6. slot-tagger parser with Stage 3

Each test case is executed in a newly generated world using the fixed evaluation seed 205. So every configuration has the same initial objects, attributes, locations, and relations. Test-specific setup commands are executed with the rule-based parser so that cases requiring a particular world state can be evaluated consistently. The evaluated command is then processed with the selected parser (and the Stage 3 model checker, when selected).

The evaluation compares each system result with the expected gold-standard interpretation regarding:
* **intent accuracy**: whether the predicted command intent is correct
* **grounded-frame exact match**: whether the intent and grounded argument fields match the expected frame
* **resolution exact match**: whether object, tool, and target resolution states and candidate sets are correct
* **execution exact match**: whether the final execution status and success value are correct
* **end-to-end exact match**: whether grounding, resolution, and execution are all correct for the same command
* **technical error count**: the number of cases that terminate with an unexpected exception

Candidate lists are compared as sets, so the order does not affect the result. Metrics are calculated for the complete evaluation set and separately for each command category.

For Stage 3 configurations, the notebook also records how often accepted model corrections were helpful or harmful. Intent corrections are assessed by comparing the original and final intent predictions, while accepted object and tool selections are checked against the expected grounded IDs. Stage 3 uses a confidence threshold of 0.90. Lower-confidence suggestions are rejected in favor of the original symbolic result.

The detailed output of every configuration is saved as JSON. The aggregated metrics and Stage 3 correction analysis are saved as CSV files in `evaluation_results/end_to_end_results`.


In [1]:
import os
from getpass import getpass

hf_token = os.getenv("HF_TOKEN") or getpass(
    "Enter your Hugging Face token: "
)

Enter your Hugging Face token:  ········


In [2]:
import sys
import json
import pandas as pd
from pathlib import Path
from typing import Optional, Dict, Any, FrozenSet


PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import dialogue, parsers, stage3, world

EVALUATION_SEED = 205

EVALUATION_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "evaluation_data"
    / "evaluation_scenarios.json"
)
RESULT_DIR = (
    PROJECT_ROOT
    / "evaluation_results"
    / "end_to_end_results"
)

METRICS_PATH = RESULT_DIR / "metrics.csv"
STAGE3_PATH = RESULT_DIR / "stage3_corrections.csv"

In [3]:
# Fields retained from each interpretation result for later evaluation
RESULT_FIELDS = [
    "intent",
    "original_intent",
    "object_id",
    "tool_id",
    "next_to_id",
    "location",
    "tool_type",
    "created_object_id",
    "status",
    "success",
    "object_resolution",
    "tool_resolution",
    "target_resolution",
    "object_candidate_ids",
    "tool_candidate_ids",
    "target_candidate_ids",
    "model_corrected_intent",
    "model_corrected_object",
    "model_corrected_tool"
]



#-----------------
# Helper functions
#-----------------

def serialize_interpretation_result(
    result: dialogue.InterpretationResult,
) -> dict:
    """Convert an interpretation result into a JSON-serializable dictionary."""
    
    return {
        field_name: getattr(result, field_name)
        for field_name in RESULT_FIELDS
    }



def run_test_case(
    test_case: dict,
    parser: parsers.BaseParser,
    model_checker: Optional[stage3.ModelChecker] = None,
) -> Dict[str, Any]:
    """Run one evaluation case in a fresh reproducible world."""
    
    # Reset the world so every test case starts from the same initial state
    evaluation_world = world.World.create_random(seed=EVALUATION_SEED)
    setup_results = []

    try:
        # Apply required state changes before evaluating the main command
        for setup_command in test_case.get("setup_commands", []):
            setup_parser = parsers.RuleBasedParser()
            setup_result = dialogue.interpret_and_act(
                world=evaluation_world,
                utterance=setup_command,
                parser=setup_parser,
                model_checker=None,
                intent_display=False
            )
            setup_results.append({
                "command": setup_command,
                "result": serialize_interpretation_result(setup_result),
            })

            # Do not evaluate the main command if its required setup failed
            if setup_result.status != "EXECUTED" or setup_result.success is not True:
                return {
                    "id": test_case["id"],
                    "command": test_case["command"],
                    "setup_results": setup_results,
                    "result": None,
                    "error": f"Setup failed for command: {setup_command}",
                }

        result = dialogue.interpret_and_act(
            world=evaluation_world,
            utterance=test_case["command"],
            parser=parser,
            model_checker=model_checker,
            intent_display=False
        )

        # Record technical failures without interrupting the remaining cases
        return {
            "id": test_case["id"],
            "command": test_case["command"],
            "setup_results": setup_results,
            "result": serialize_interpretation_result(result),
            "error": None,
        }

    except Exception as error:
        return {
            "id": test_case["id"],
            "command": test_case["command"],
            "setup_results": setup_results,
            "result": None,
            "error": f"{type(error).__name__}: {error}",
        }



def run_parser_evaluation(
    dataset_path: str,
    output_path: str,
    parser_name: str,
    parser: parsers.BaseParser,
    model_checker: Optional[stage3.ModelChecker] = None,
) -> None:
    """Evaluate one parser configuration and save its detailed results."""
    
    with open(dataset_path, "r", encoding="utf-8") as file:
        test_cases = json.load(file)
    results = [
        run_test_case(
            test_case=test_case,
            parser=parser,
            model_checker=model_checker
        )
        for test_case in test_cases
    ]
    output = {
        "parser_name": parser_name,
        "model_check": model_checker is not None,
        "evaluation_seed": EVALUATION_SEED,
        "number_of_cases": len(results),
        "results": results
    }
    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    # Preserve detailed per-case results for later metric calculation
    with output_file.open("w", encoding="utf-8") as file:
        json.dump(output, file, indent=2, ensure_ascii=False)



#----------------------
# Run on parser configs
#----------------------

# Use a strict confidence threshold for accepting Stage 3 corrections
model_checker = stage3.ModelChecker(
    hf_token=hf_token,
    verbose=False, 
    confidence_threshold=0.90
)

# Evaluate each parser both independently and with Stage 3 model checking
parser_configs = [
    {
        "name": "rule_based",
        "parser": parsers.RuleBasedParser(),
        "model_checker": None,
    },
    {
        "name": "rule_based_stage3",
        "parser": parsers.RuleBasedParser(),
        "model_checker": model_checker,
    },
    {
        "name": "intent_classifier",
        "parser": parsers.IntentClassifierParser(),
        "model_checker": None,
    },
    {
        "name": "intent_classifier_stage3",
        "parser": parsers.IntentClassifierParser(),
        "model_checker": model_checker,
    },
    {
        "name": "slot_tagger",
        "parser": parsers.SlotTaggerParser(),
        "model_checker": None,
    },
    {
        "name": "slot_tagger_stage3",
        "parser": parsers.SlotTaggerParser(),
        "model_checker": model_checker,
    },
]

for config in parser_configs:
    run_parser_evaluation(
        dataset_path=EVALUATION_DATA_PATH,
        output_path=(
            RESULT_DIR / f"{config['name']}.json"
        ),
        parser_name=config["name"],
        parser=config["parser"],
        model_checker=config["model_checker"],
    )

In [4]:
# Result files produced by the evaluated parser configurations
RESULT_FILES = [
    RESULT_DIR / "rule_based.json",
    RESULT_DIR / "rule_based_stage3.json",
    RESULT_DIR / "intent_classifier.json",
    RESULT_DIR / "intent_classifier_stage3.json",
    RESULT_DIR / "slot_tagger.json",
    RESULT_DIR / "slot_tagger_stage3.json"
]



# ----------------
# Evaluated fields
# ----------------

# Fields describing the predicted grounded command frame
FRAME_FIELDS = [
    "intent",
    "object_id",
    "tool_id",
    "location",
    "next_to_id"
]
# Fields describing whether command execution succeede
EXECUTION_FIELDS = [
    "status",
    "success"
]
# Reference roles evaluated for resolution quality and candidate selection
RESOLUTION_ROLES = [
    "object",
    "tool",
    "target"
]
RESOLUTION_FIELDS = [
    f"{role}_resolution"
    for role in RESOLUTION_ROLES
]
CANDIDATE_FIELDS = [
    f"{role}_candidate_ids"
    for role in RESOLUTION_ROLES
]
# Complete set of fields used for detailed result comparison
COMPARED_FIELDS = (FRAME_FIELDS + EXECUTION_FIELDS + RESOLUTION_FIELDS + CANDIDATE_FIELDS)
# Flags indicating that Stage 3 replaced an earlier parser or resolver result
CORRECTION_FLAGS = [
    "model_corrected_intent",
    "model_corrected_object",
    "model_corrected_tool",
]
# Explicit placeholder used when normalizing missing values for comparison
NULL_VALUE = "__NULL__"



# ---------------------------
# Comparison helper functions
# ---------------------------

def scalar_match(
    data: pd.DataFrame,
    field: str,
) -> pd.Series:
    """Compare one expected and actual scalar field."""

    # Replace missing values with the same explicit marker before comparison
    expected = (
        data[f"expected_{field}"]
        .astype("object")
        .where(data[f"expected_{field}"].notna(), NULL_VALUE)
    )
    actual = (
        data[f"actual_{field}"]
        .astype("object")
        .where(data[f"actual_{field}"].notna(), NULL_VALUE)
    )
    return expected.eq(actual)


    
def all_scalar_fields_match(
    data: pd.DataFrame,
    fields: list[str],
) -> pd.Series:
    """Require all listed scalar fields to match."""
    matches = [
        scalar_match(data, field)
        for field in fields
    ]

    # Combine the field-wise results and require every field to match per row
    return pd.concat(matches, axis=1).all(axis=1)

    

def as_candidate_set(value: Any) -> FrozenSet[Any]:
    """Convert a candidate collection into an order-independent frozen set.

    Values that are not list-like candidate collections are treated as empty.
    """
    
    if isinstance(value, (list, tuple, set, frozenset)):
        return frozenset(value)
    return frozenset()

    

def candidate_set_match(data: pd.DataFrame, field: str) -> pd.Series:
    """Compare expected and actual candidate collections without considering order."""
    
    expected = data[f"expected_{field}"].map(as_candidate_set)
    actual = data[f"actual_{field}"].map(as_candidate_set)
    return expected.eq(actual)

    

def resolution_role_match(data: pd.DataFrame, role: str) -> pd.Series:
    """Require both the resolution state and candidate set to match."""
    
    return (
        scalar_match(data, f"{role}_resolution")
        & candidate_set_match(data, f"{role}_candidate_ids")
    )



#---------------------------------------------------
# Main metrics calculation and preparation functions
# --------------------------------------------------

def calculate_metrics(data: pd.DataFrame) -> Dict[str, int | float]:
    """Calculate aggregate performance metrics and technical error counts."""
    
    intent_correct = scalar_match(data, "intent")
    frame_correct = all_scalar_fields_match(data, FRAME_FIELDS)

    # A resolution is correct only when every reference role matches
    resolution_correct = pd.concat(
        [
            resolution_role_match(data, role)
            for role in RESOLUTION_ROLES
        ],
        axis=1
    ).all(axis=1)
    execution_correct = all_scalar_fields_match(data, EXECUTION_FIELDS)

    # End-to-end success requires correct grounding, resolution, and execution
    end_to_end_correct = (
        frame_correct
        & resolution_correct
        & execution_correct
    )
    return {
        "n_cases": len(data),
        "intent_accuracy": intent_correct.mean(),
        "grounded_frame_exact_match": frame_correct.mean(),
        "resolution_exact_match": resolution_correct.mean(),
        "execution_exact_match": execution_correct.mean(),
        "end_to_end_exact_match": end_to_end_correct.mean(),
        "technical_error_count": int(data["error"].notna().sum()),
    }



def calculate_stage3_summary(data: pd.DataFrame) -> Dict[str, int]:
    """Summarize the effect of accepted Stage 3 corrections."""

    final_intent_correct = scalar_match(data, "intent")

    # Compare the parser's original intent with the expected intent 
    # before considering accepted corrections from Stage 3
    original_intent_correct = (
        data["actual_original_intent"]
        .astype("object")
        .where(data["actual_original_intent"].notna(), NULL_VALUE)
        .eq(
            data["expected_intent"]
            .astype("object")
            .where(data["expected_intent"].notna(), NULL_VALUE)
        )
    )

     # Missing correction flags mean that no Stage 3 correction was accepted
    intent_changed = (
        data["actual_model_corrected_intent"]
        .fillna(False)
        .astype(bool)
    )
    object_changed = (
        data["actual_model_corrected_object"]
        .fillna(False)
        .astype(bool)
    )
    tool_changed = (
        data["actual_model_corrected_tool"]
        .fillna(False)
        .astype(bool)
    )

    # Corrections are helpful if they change an incorrect intent to a correct one
    intent_helpful = (
        intent_changed
        & ~original_intent_correct
        & final_intent_correct
    )
    # Corrections are harmful if they change an correct intent to a incorrect one
    intent_harmful = (
        intent_changed
        & original_intent_correct
        & ~final_intent_correct
    )

    # Object/tool selections are evaluated only when the final intent is correct
    # Reason: the intended argument roles depend on the command intent
    object_evaluated = (object_changed & final_intent_correct)
    object_correct = (object_evaluated & scalar_match(data, "object_id"))
    tool_evaluated = (tool_changed & final_intent_correct)
    tool_correct = (tool_evaluated & scalar_match(data, "tool_id"))
    
    return {
        "intent_changes": int(intent_changed.sum()),
        "intent_helpful": int(intent_helpful.sum()),
        "intent_harmful": int(intent_harmful.sum()),
        "object_changes_evaluated": int(object_evaluated.sum()),
        "object_correct": int(object_correct.sum()),
        "object_wrong": int(
            (object_evaluated & ~object_correct).sum()
        ),
        "tool_changes_evaluated": int(tool_evaluated.sum()),
        "tool_correct": int(tool_correct.sum()),
        "tool_wrong": int(
            (tool_evaluated & ~tool_correct).sum()
        )
    }



def build_evaluation_frame(
    result_data: dict,
    gold_cases: list[dict],
) -> pd.DataFrame:
    """Align one parser result file with the corresponding gold cases."""

    # Index results by case ID so they can be aligned independently of order
    results_by_id = {
        item["id"]: item
        for item in result_data["results"]
    }
    rows = []
    for gold_case in gold_cases:
        case_id = gold_case["id"]
        expected = gold_case["expected"]
        # Missing or failed result cases are represented by empty actual values
        result_case = results_by_id.get(case_id, {})
        actual = result_case.get("result") or {}
        
        row = {
            "id": case_id,
            "category": gold_case["category"],
            "command": gold_case["command"],
            "error": result_case.get("error"),
            "actual_original_intent": actual.get(
                "original_intent"
            )
        }
        # Store expected and actual values in paired columns for comparison
        for field in COMPARED_FIELDS:
            row[f"expected_{field}"] = expected.get(field)
            row[f"actual_{field}"] = actual.get(field)
        for flag in CORRECTION_FLAGS:
            row[f"actual_{flag}"] = actual.get(flag, False)
            
        rows.append(row)
        
    return pd.DataFrame(rows)



# ----------------------------------
# Evaluate all parser configurations
# ----------------------------------

# Load gold evaluation data
with EVALUATION_DATA_PATH.open("r", encoding="utf-8") as file:
    gold_cases = json.load(file)

metric_rows = []
stage3_rows = []
for result_file in RESULT_FILES:
    if not result_file.exists():
        print(f"Skipping missing file: {result_file}")
        continue
        
    with result_file.open("r", encoding="utf-8") as file:
        result_data = json.load(file)
        
    evaluation_df = build_evaluation_frame(result_data, gold_cases)
    parser_name = result_data["parser_name"]
    model_check = result_data.get("model_check", False)

    # Calculate metrics for the full dataset and for each command category
    groups = [
        ("ALL", evaluation_df),
        *list(evaluation_df.groupby("category"))
    ]
    for category, category_data in groups:
        metric_rows.append({
            "parser_name": parser_name,
            "model_check": model_check,
            "category": category,
            **calculate_metrics(category_data)
        })

    # Correction statistics apply only to configurations using Stage 3
    if model_check:
        stage3_rows.append({
            "parser_name": parser_name,
            "model_check": model_check,
            **calculate_stage3_summary(evaluation_df)
        })


metrics_df = pd.DataFrame(metric_rows)
stage3_df = pd.DataFrame(stage3_rows)
metrics_df.to_csv(METRICS_PATH, index=False)
stage3_df.to_csv(STAGE3_PATH, index=False)

display(metrics_df)
display(stage3_df)

,parser_name,model_check,category,n_cases,intent_accuracy,grounded_frame_exact_match,resolution_exact_match,execution_exact_match,end_to_end_exact_match,technical_error_count
0,rule_based,False,ALL,70,0.928571,0.842857,0.871429,0.885714,0.842857,0
1,rule_based,False,basic,20,1.000000,1.000000,1.000000,1.000000,1.000000,0
2,rule_based,False,complex_reference,15,0.933333,0.666667,0.666667,0.733333,0.666667,0
3,rule_based,False,natural_language_variation,20,0.800000,0.700000,0.800000,0.800000,0.700000,0
4,rule_based,False,truly_ambiguous_or_unresolvable,15,1.000000,1.000000,1.000000,1.000000,1.000000,0
5,rule_based_stage3,True,ALL,70,0.985714,0.785714,0.785714,0.857143,0.785714,0
6,rule_based_stage3,True,basic,20,1.000000,1.000000,1.000000,1.000000,1.000000,0
7,rule_based_stage3,True,complex_reference,15,1.000000,0.800000,0.800000,1.000000,0.800000,0
8,rule_based_stage3,True,natural_language_variation,20,0.950000,0.950000,0.950000,0.950000,0.950000,0
9,rule_based_stage3,True,truly_ambiguous_or_unresolvable,15,1.000000,0.266667,0.266667,0.400000,0.266667,0


,parser_name,model_check,intent_changes,intent_helpful,intent_harmful,object_changes_evaluated,object_correct,object_wrong,tool_changes_evaluated,tool_correct,tool_wrong
0,rule_based_stage3,True,4,4,0,13,3,10,3,2,1
1,intent_classifier_stage3,True,0,0,0,12,3,9,3,2,1
2,slot_tagger_stage3,True,0,0,0,11,3,8,2,1,1
